# Phase 2 — NSE Data Acquisition

Acquires and validates NSE company master, market data, and financial filing data.

In [ ]:
from datetime import date
from pathlib import Path

from src.data.providers.nse_security_master import build_security_master
from src.data.providers.nse_market import collect_market_data, save_market_data
from src.data.providers.nse_financials import (
    collect_filing_metadata,
    acquire_financial_filings,
)
from src.data.processing.market_processing import integrate_market_with_company_master
from src.data.processing.financial_normalization import (
    combine_filing_data,
    validate_filing_dataset,
)

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "stock-investment-ml":
    PROJECT_DIR = Path("/content/stock-investment-ml")

print("Project:", PROJECT_DIR)

In [ ]:
# Build company master and investable EQ universe

company_master = build_security_master()

investable_universe = (
    company_master[company_master["series"].eq("EQ")]
    .reset_index(drop=True)
)

print("Company master:", len(company_master))
print("EQ universe:", len(investable_universe))

In [ ]:
# Acquire and integrate NSE market data

market_data = collect_market_data(
    date(2026, 8, 18),
    date(2026, 8, 22),
)

company_prices = integrate_market_with_company_master(
    market_data,
    investable_universe,
)

market_path = save_market_data(company_prices)

print("Market rows:", len(company_prices))
print("Companies:", company_prices["isin"].nunique())
print("Saved:", market_path)

In [ ]:
# Acquire financial filing metadata

filing_records = collect_filing_metadata(
    page_size=100,
    max_pages=1,
)

print("Financial filings acquired:", len(filing_records))

In [ ]:
# Acquire XBRL financial data

(
    financial_metadata_df,
    financial_facts_df,
    financial_contexts_df,
    financial_manifest_df,
) = acquire_financial_filings(filing_records)

validate_filing_dataset(
    financial_metadata_df,
    financial_facts_df,
    financial_contexts_df,
)

print("Filings:", len(financial_metadata_df))
print("Facts:", len(financial_facts_df))
print("Contexts:", len(financial_contexts_df))

In [ ]:
# Combine filing data

financial_dataset = combine_filing_data(
    financial_metadata_df,
    financial_facts_df,
    financial_contexts_df,
)

print("Financial dataset:", financial_dataset.shape)

In [ ]:
# Save processed financial datasets

output_dir = PROJECT_DIR / "data" / "processed" / "nse" / "financials"
output_dir.mkdir(parents=True, exist_ok=True)

financial_metadata_df.to_parquet(
    output_dir / "filing_metadata.parquet",
    index=False,
)

financial_facts_df.to_parquet(
    output_dir / "xbrl_facts.parquet",
    index=False,
)

financial_contexts_df.to_parquet(
    output_dir / "xbrl_contexts.parquet",
    index=False,
)

financial_dataset.to_parquet(
    output_dir / "financial_dataset.parquet",
    index=False,
)

financial_manifest_df.to_json(
    output_dir / "financial_downloads.json",
    orient="records",
    indent=2,
)

print("Financial data saved to:", output_dir)

## Acquisition Complete

This notebook performs data acquisition and structural validation only.

Financial metric interpretation, historical expansion, feature engineering, target creation, machine learning, ranking, and backtesting are handled in later phases.